In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [5]:
!pip install crewai; openai; google-search-results; crewai_tools

  Using cached crewai-1.14.6-py3-none-any.whl.metadata (36 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached crewai_cli-1.14.6-py3-none-any.whl.metadata (1.7 kB)
  Using cached crewai_core-1.14.6-py3-none-any.whl.metadata (1.2 kB)
  Using cached instructor-1.15.1-py3-none-any.whl.metadata (12 kB)
  Using cached json_repair-0.25.3-py3-none-any.whl.metadata (7.9 kB)
  Using cached json5-0.10.0-py3-none-any.whl.metadata (34 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached lancedb-0.30.0-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached mcp-1.26.0-py3-none-any.whl.metadata (89 kB)
  Using cached opentelemetry_api-1.34.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached op

In [2]:
import os
from crewai import Agent, Task, Crew
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import userdata


#api_key = os.getenv("OPENAI_API_KEY")

## Use below if you are using google collab
api_key = userdata.get('OPENAI_API_KEY')

In [3]:
# Define Customer Service agent
customer_assist_agent = Agent(
    role="Customer Care Specialist",
    goal="Deliver the most correct and effective "
         "customer support on your team.",
    backstory=(
        "You are a valued team member at B-MAD (https://docs.bmad-method.org/) "
        "and currently assigned to assist {customer}, a crucial client "
        "for the company."
        "Your mission is to ensure the highest quality of support!"
        "Provide detailed and thorough responses, "
        "avoiding any assumptions."
    ),
    allow_delegation=False,
    verbose=True
)

In [4]:
#Define second agent(Level 2)
quality_check_agent = Agent(
    role="Support Quality Analyst",
    goal="Be recognized for ensuring the highest "
         "standards of support quality in your team.",
    backstory=(
        "You are a key contributor at  B-MAD (https://docs.bmad-method.org/) , "
        "working alongside your team to review a support request from {customer}. "
        "Your responsibility is to ensure that the Customer Care Specialist "
        "delivers top-notch service.\n"
        "It’s your duty to confirm that all responses are thorough, "
        "accurate, and free of assumptions."
    ),
    verbose=True
)

In [10]:
!pip install crewai-tools;

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.6/809.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 39.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.13.0
    Uninstalling tiktoken-0.13.0:
      Successfully uninstalled tiktoken-0.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavio

In [5]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.bmad-method.org/",config={}
)

customer_query_task = Task(
    description=(
        "{customer} has submitted a critical request:\n"
        "{inquiry}\n\n"
        "{person} from {customer} initiated the inquiry. "
        "Leverage all available resources and knowledge to "
        "deliver outstanding support."
        "Your goal is to provide a thorough and accurate resolution "
        "to the customer's question."
    ),
    expected_output=(
        "A comprehensive and well-researched response to the customer's inquiry "
        "that addresses every aspect of their concern.\n"
        "The response must reference all the tools and information sources used, "
        "including external materials or solutions. "
        "Ensure that the answer is exhaustive, leaves no room for follow-up questions, "
        "and maintains a professional, approachable tone."
    ),
    tools=[docs_scrape_tool],
    agent=customer_assist_agent,
)

response_review_task = Task(
    description=(
        "Evaluate the response prepared by the Customer Care Specialist for {customer}'s inquiry. "
        "Ensure the reply meets the highest standards of accuracy, completeness, and clarity "
        "expected in customer service.\n"
        "Confirm that every aspect of the customer's question has been addressed "
        "in a friendly and approachable manner.\n"
        "Verify the inclusion of appropriate references and sources used "
        "to gather the information, ensuring the response is well-supported and "
        "leaves no loose ends."
    ),
    expected_output=(
        "A polished, detailed, and ready-to-send response that completely addresses "
        "the customer's inquiry.\n"
        "The response should reflect all necessary feedback and adjustments while "
        "maintaining a relaxed yet professional tone that aligns with our cool and casual company culture."
    ),
    agent=quality_check_agent,
)

In [6]:
crewBMADAssistance = Crew(
    agents=[customer_assist_agent, quality_check_agent],
    tasks=[customer_query_task, response_review_task],
    verbose= 2,
    memory= True
)

inputs = {
    "customer": "XYZ Development Company",
    "person": "Vishwanjali Jadhav",
    "inquiry": "I need help with B-MAD Method "
               "and Understanding about B-MAD and Agents "
               "How to use B-MAD for architecture documentation"
               "Can you provide guidance?"
}
result = crewBMADAssistance.kickoff(inputs=inputs)

ValidationError: 1 validation error for Crew
verbose
  Input should be a valid boolean, unable to interpret input [type=bool_parsing, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/bool_parsing